# LLM Fine-Tuning Walkthrough (Live Demo)

This notebook walks through the **same pipeline** as the `scripts/` folder, but interactively so you can run each snippet live and explain it.

**Pipeline:**

```
data/docs/*.txt  ->  chunks.jsonl  ->  train.jsonl  ->  fine-tuned model (out_model)  ->  generate / inspect
```

Run the cells top to bottom. Each section has a short **Concept** note and a **Checkpoint** question for the audience.

## 0. Setup

Locate the repo, import libraries. If you launched Jupyter from the `model_training` folder this just works.

In [ ]:
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Resolve repo root whether we run from notebooks/ or model_training/
CWD = Path.cwd()
REPO = CWD.parent if CWD.name == 'notebooks' else CWD
DATA = REPO / 'data'
OUT_MODEL = REPO / 'out_model'

print('Repo root :', REPO)
print('Data dir  :', DATA, '(exists:', DATA.exists(), ')')
print('out_model :', OUT_MODEL, '(exists:', OUT_MODEL.exists(), ')')

## 1. Raw documents -> Chunks  (`prepare_data.py`)

**Concept:** LLMs train on bounded pieces of text. We split each document into paragraphs (split on blank lines) and store one JSON record per paragraph in `chunks.jsonl`.

Let's look at the source docs and the chunks produced from them.

In [ ]:
# Show the source documents
for f in sorted((DATA / 'docs').glob('*.txt')):
    text = f.read_text(encoding='utf-8')
    print(f'--- {f.name}  ({len(text)} chars) ---')
    print(text[:300], '...\n' if len(text) > 300 else '\n')

In [ ]:
# Show the first few chunks (id + source + text)
with open(DATA / 'chunks.jsonl', encoding='utf-8') as fh:
    chunks = [json.loads(line) for line in fh]

print('Total chunks:', len(chunks), '\n')
for c in chunks[:3]:
    print('id    :', c['id'][:12], '...')
    print('source:', c['source'])
    print('text  :', c['text'][:160], '...' if len(c['text']) > 160 else '')
    print()

> **Checkpoint 1:** Why chunk instead of feeding whole documents? Models have a max context length, and smaller units give more, more-varied training examples. The `id` is a SHA-1 hash so identical text always gets the same fingerprint.

## 2. Chunks -> Training data  (`prepare_lm_data.py`)

**Concept:** This is **self-supervised** data. There are no labels — the 'answer' is just the next word in the text. We store it as `train.jsonl` (`{"text": ...}` per line).

In [ ]:
with open(DATA / 'lm' / 'train.jsonl', encoding='utf-8') as fh:
    train_rows = [json.loads(line) for line in fh]

print('Training examples:', len(train_rows), '\n')
for r in train_rows[:3]:
    print('-', r['text'][:140], '...' if len(r['text']) > 140 else '')

> **Checkpoint 2:** Where do the labels come from if we never wrote any? In language modeling the label for each token IS the next token — the text supervises itself.

## 3. Tokenization  (the bridge between text and the model)

**Concept:** The tokenizer converts text <-> integer IDs. GPT-2 uses **subword** tokens (byte-pair encoding), so words can be split into pieces. The leading `Ġ` marker means 'space before this token'.

We load from `out_model` (your fine-tuned model). If it isn't there, we fall back to `distilgpt2`.

In [ ]:
MODEL_REF = str(OUT_MODEL) if OUT_MODEL.exists() else 'distilgpt2'
print('Loading tokenizer from:', MODEL_REF)

tokenizer = AutoTokenizer.from_pretrained(MODEL_REF, use_fast=True)
print('Vocab size:', tokenizer.vocab_size)

sample = 'This is a short document about cooking pasta.'
enc = tokenizer(sample, return_tensors='pt')
input_ids = enc['input_ids']

print('\nText      :', sample)
print('Input IDs :', input_ids.tolist())
print('Tokens    :', tokenizer.convert_ids_to_tokens(input_ids[0].tolist()))
print('Length    :', input_ids.shape[1], 'tokens')

> **Checkpoint 3:** Notice the model's unit is *subwords*, not words. Try editing `sample` above to a made-up word like `'antidisestablishmentarianism'` and re-run — watch it split into multiple tokens.

## 4. The model: logits -> probabilities -> next token  (`inspect_model.py`)

**Concept:** For every position the model outputs a **logit** (raw score) for *every* token in the ~50k vocabulary. `softmax` turns those scores into probabilities that sum to 1. The model's whole job is: 'given the text so far, what comes next?'

We load with `attn_implementation='eager'` so we can also see attention weights later.

In [ ]:
print('Loading model from:', MODEL_REF)
model = AutoModelForCausalLM.from_pretrained(MODEL_REF, attn_implementation='eager')
model.eval()

with torch.no_grad():
    outputs = model(**enc, output_attentions=True)

logits = outputs.logits
print('Logits shape:', tuple(logits.shape), '= (batch, seq_len, vocab_size)')

In [ ]:
# Next-token distribution for the LAST token of the prompt
last_logits = logits[0, -1, :]
probs = torch.softmax(last_logits, dim=-1)
topk = torch.topk(probs, k=10)

print('Top-10 predicted next tokens after:', repr(sample), '\n')
for idx, p in zip(topk.indices.tolist(), topk.values.tolist()):
    print(f'  {tokenizer.decode([idx])!r:>12} : {p:.4f}')

> **Checkpoint 4:** The model produced ~50,000 numbers for this one position. After softmax they form a probability distribution. The token with the highest probability is what 'greedy' generation would pick.

## 5. Attention: which tokens look at which  (`inspect_model.py`)

**Concept:** A Transformer layer lets each token 'attend to' earlier tokens. Each layer has multiple heads. The matrix below (averaged over heads of the last layer) shows how much each row token attends to each column token. Because this is a *causal* model, tokens can only attend backward (note the lower-triangular pattern).

In [ ]:
attentions = outputs.attentions
if attentions:
    print('Attention layers:', len(attentions))
    print('Each layer shape:', tuple(attentions[0].shape), '= (batch, heads, seq, seq)\n')

    last_layer = attentions[-1][0]      # (heads, seq, seq)
    avg_head = last_layer.mean(0)        # (seq, seq)
    toks = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    header = ' ' * 12 + ''.join(f'{t[:7]:>8}' for t in toks)
    print('Averaged attention (last layer). Rows = query, Cols = key:\n')
    print(header)
    for t, row in zip(toks, avg_head.tolist()):
        print(f'{t[:11]:>11} ' + ''.join(f'{v:8.3f}' for v in row))
else:
    print('No attention returned. Ensure the model loaded with attn_implementation="eager".')

Optional: a heatmap makes the lower-triangular (causal) pattern obvious. Requires `matplotlib`.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(avg_head.numpy(), cmap='viridis')
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90)
    ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks)
    ax.set_title('Averaged attention (last layer)')
    fig.colorbar(im)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed - skip heatmap (pip install matplotlib)')

> **Checkpoint 5:** Why is the attention matrix lower-triangular? Because a causal LM can only look at past tokens, never future ones — that's what makes it generate left-to-right.

## 6. Generation: predict-next-token in a loop  (`generate.py`)

**Concept:** Generation is just step 4 repeated: predict a token, append it, feed it back, repeat. The **decoding strategy** controls *how* we pick each token.

We pass an `attention_mask` to avoid the warning you saw on the command line (pad token == eos token).

In [ ]:
def generate(prompt, max_new_tokens=60, do_sample=False, temperature=0.9, top_k=50, top_p=0.95, seed=42):
    inputs = tokenizer(prompt, return_tensors='pt')
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        torch.manual_seed(seed)
        gen_kwargs.update(temperature=temperature, top_k=top_k, top_p=top_p)
    with torch.no_grad():
        out_ids = model.generate(
            inputs['input_ids'], attention_mask=inputs['attention_mask'], **gen_kwargs
        )
    text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

In [ ]:
prompt = 'How to make pasta'

print('=== GREEDY (do_sample=False) ===')
print(generate(prompt, do_sample=False))

print('\n=== SAMPLING (temperature=0.9) ===')
print(generate(prompt, do_sample=True, temperature=0.9))

**Try it live:** change `prompt` and the `temperature`. Low temperature (e.g. 0.3) = focused/repetitive; high (e.g. 1.3) = creative/random.

> **Checkpoint 6:** Greedy always picks the single most-likely token, which causes the looping you may see. Sampling adds controlled randomness so output is more varied.

## 7. Key takeaways

- A fine-tuned model = **big pretrained brain + small custom nudge**. With a tiny dataset the pretrained knowledge dominates.
- The model is **self-contained and offline** after the one-time weight download — no live internet calls during generation.
- It generates from **statistical patterns**, not facts, so it can hallucinate confidently.
- To ground answers in *your* documents you'd either fine-tune much more, or build a **RAG** system (retrieve your chunks at query time) — planned for a future session.